In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import shutil

from glam import Geocoder, download_dependencies
from pathlib import Path

import geopandas as gpd

## Checkpoint: load cached processed data if available

Skips straight to a df with addresses parsed, geocoded, and hazard-flagged —
avoids re-running the ~30min geocoding pass. If this loads successfully, skip
ahead to the "Analysis" section below rather than re-running the parsing /
geocoding / hazard cells.

In [6]:
CACHE_PATH = Path("./cache/dvrs_processed.parquet")

if CACHE_PATH.exists():
    df = pd.read_parquet(CACHE_PATH)
    print(f"Loaded cached df from {CACHE_PATH}, shape={df.shape}")
else:
    df = None
    print("No cache found — run the full pipeline below, then save a checkpoint at the end.")


Loaded cached df from cache\dvrs_processed.parquet, shape=(90738, 52)


In [3]:
import csv
import io

with open("DVRS270826.txt", encoding="utf-16") as f:
    lines = f.read().splitlines()

header, *rows = lines
expected_cols = len(header.split("|"))


def unwrap(line):
    if len(line) >= 2 and line[0] == '"' and line[-1] == '"':
        return line[1:-1]
    return line


cleaned = [unwrap(line) for line in rows]

# Drop genuinely malformed rows (e.g. a stray "#NAME?" Excel-error row) instead
# of letting them silently misalign columns.
good_rows = [line for line in cleaned if len(line.split("|")) == expected_cols]
n_dropped = len(cleaned) - len(good_rows)
if n_dropped:
    print(f"Dropping {n_dropped} malformed row(s) that don't split into {expected_cols} fields")

csv_text = "\n".join([header] + good_rows)
df = pd.read_csv(io.StringIO(csv_text), sep="|", dtype=str, quoting=csv.QUOTE_NONE)


Dropping 1 malformed row(s) that don't split into 41 fields


In [4]:
# Normalize column names

df.columns = (
    df.columns
    .str.strip()
    .str.lower()
    .str.replace(r"[^a-z0-9]+", "_", regex=True)
    .str.strip("_")
)

In [5]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 92010 entries, 0 to 92009
Data columns (total 41 columns):
 #   Column                            Non-Null Count  Dtype
---  ------                            --------------  -----
 0   valuation_roll_number             92010 non-null  str  
 1   valuation_number_assessment       92010 non-null  str  
 2   valuation_number_suffix           14153 non-null  str  
 3   sale_date                         92010 non-null  str  
 4   district_code                     92010 non-null  str  
 5   sale_type                         92009 non-null  str  
 6   sales_group                       92010 non-null  str  
 7   sale_tenure                       92010 non-null  str  
 8   price_value_relationship          92010 non-null  str  
 9   sale_price_gross                  92010 non-null  str  
 10  sale_price_net                    92010 non-null  str  
 11  sale_price_chattels               92010 non-null  str  
 12  sale_price_other                  92010 non

In [6]:
# Sanity check
print(df.shape)
print(df[["sale_date", "sale_price_gross", "sale_price_net"]].isna().sum())

(92010, 41)
sale_date           0
sale_price_gross    0
sale_price_net      0
dtype: int64


## Geocode via LINZ Address Matching

In [4]:
def download_glam_dependencies(deps_directory: str) -> Path:
    deps_path = Path(deps_directory)
    marker = deps_path / "nz-street-address.csv"

    if marker.exists():
        print(f"glam dependencies already present in {deps_path}, skipping download")
    else:
        download_dependencies(str(deps_path))


In [ ]:
# Limit the geocoder's candidate pool to Auckland-region addresses only.
# Every property here is confirmed within Auckland (district_code == 14), so
# any match outside the region is unambiguously wrong (we saw e.g. a match
# to "Mosgiel", ~1000km away in Dunedin, with the full-NZ reference data).
# Restricting the candidates is more reliable than nudging the query text
# with a token like "Auckland", which Papakura/Pukekohe/Manurewa etc. don't
# consistently carry in their address.
AKL_LON_MIN, AKL_LON_MAX = 174.1728, 175.5419
AKL_LAT_MIN, AKL_LAT_MAX = -37.2966, -36.0497

national_deps_dir = Path("./glam-deps")
if not (national_deps_dir / "nz-street-address.csv").exists():
    download_glam_dependencies(str(national_deps_dir))

deps_dir = Path("./glam-deps-auckland")
if not (deps_dir / "nz-street-address.csv").exists():
    nzsa_national = pd.read_csv(national_deps_dir / "nz-street-address.csv", dtype=str)
    nzsa_national["shape_X"] = nzsa_national["shape_X"].astype(float)
    nzsa_national["shape_Y"] = nzsa_national["shape_Y"].astype(float)

    nzsa_akl = nzsa_national[
        nzsa_national["shape_X"].between(AKL_LON_MIN, AKL_LON_MAX)
        & nzsa_national["shape_Y"].between(AKL_LAT_MIN, AKL_LAT_MAX)
    ]
    print(f"{len(nzsa_akl)} of {len(nzsa_national)} national addresses fall inside the Auckland bounding box")

    deps_dir.mkdir(parents=True, exist_ok=True)
    nzsa_akl.to_csv(deps_dir / "nz-street-address.csv", index=False)

gc = Geocoder(str(deps_dir), matcher="tfidf", parser="rnn")

# glam bug workaround: TFIDFMatcher.build_dependencies() saves the index map
# as "idxmap.csv", but load_dependencies() looks for "idx_map.csv"
# (underscore) -- a filename mismatch in glam's own source. Build explicitly
# once here and create the alias, so the real geocoding call below never
# hits it.
if not gc.matcher.check_build():
    gc.matcher.build_dependencies()

idxmap_path = deps_dir / "matching" / "TFIDF" / "idxmap.csv"
idx_map_path = deps_dir / "matching" / "TFIDF" / "idx_map.csv"
if idxmap_path.exists() and not idx_map_path.exists():
    shutil.copyfile(idxmap_path, idx_map_path)


In [39]:
import re

# NZ Post's official street-type abbreviations (Address and layout guide),
# plus variants confirmed against actual situation_name samples from the
# cached data (e.g. "Marine PD" -> Marine Parade, "Waterloo QD" -> the real
# Waterloo Quadrant in Auckland CBD, "Onehunga ML" -> Onehunga Mall).
# Deliberately left unmapped: tokens that turned out to be Maori-name tails
# (WAI, KATARAINA, KOHU), directional suffixes (N), single repeated full
# names misread as abbreviations (PARA = "Te Ara Tu Para", HO = "Westward
# Ho"), and genuinely ambiguous ones with too little evidence (CO, TK, CY,
# CC, MI) where a wrong guess could hurt matching more than help it.
STREET_TYPE_EXPANSIONS = {
    "RD": "Road",
    "ST": "Street",
    "PL": "Place",
    "DR": "Drive",
    "AVE": "Avenue",
    "AV": "Avenue",
    "CRES": "Crescent",
    "CR": "Crescent",
    "LA": "Lane",
    "WY": "Way",
    "CL": "Close",
    "TCE": "Terrace",
    "PDE": "Parade",
    "HWY": "Highway",
    "HTS": "Heights",
    "HL": "Hill",
    "CT": "Court",
    "GR": "Grove",
    "GRV": "Grove",
    "TR": "Terrace",
    "RI": "Rise",
    "PD": "Parade",
    "LP": "Loop",
    "HW": "Highway",
    "BV": "Boulevard",
    "BVD": "Boulevard",
    "PKWY": "Parkway",
    "HT": "Heights",
    "GRNS": "Gardens",
    "QD": "Quadrant",
    "GV": "Grove",
    "ML": "Mall",
    "EP": "Esplanade",
    "GL": "Glen",
    "MW": "Mews",
    "GLD": "Glade",
    "VALL": "Valley",
    "LANE": "Lane",
    "CO": "Court",
    "CC": "Circle",
    "GNDS": "Gardens",
    "PNT": "Point",
    "TK": "Track",
    "FW": "Fairway",
    "CV": "Cove"
    
}

# Match only the trailing word of the string (word-boundary anchored) so we
# don't touch e.g. "St" when it's a leading "Saint" prefix instead of a
# trailing "Street" suffix.
_trailing_abbr_pattern = re.compile(
    r"\b(" + "|".join(sorted(STREET_TYPE_EXPANSIONS, key=len, reverse=True)) + r")$",
    re.IGNORECASE,
)


def expand_street_type(name):
    if pd.isna(name):
        return name
    return _trailing_abbr_pattern.sub(
        lambda m: STREET_TYPE_EXPANSIONS[m.group(1).upper()], name.strip()
    )


df["situation_name_expanded"] = df["situation_name"].apply(expand_street_type)

# Sanity check: how many rows actually changed
changed = (df["situation_name_expanded"] != df["situation_name"]) & df["situation_name"].notna()
print(f"{changed.sum()} of {df['situation_name'].notna().sum()} situation_name values expanded")
df.loc[changed, ["situation_name", "situation_name_expanded"]].head(10)


87400 of 90738 situation_name values expanded


,situation_name,situation_name_expanded
0,Point Chevalier RD,Point Chevalier Road
1,Montrose ST,Montrose Street
2,St Michaels AV,St Michaels Avenue
3,St Michaels AV,St Michaels Avenue
4,St Michaels AV,St Michaels Avenue
5,Raymond ST,Raymond Street
6,Harbour View RD,Harbour View Road
7,Boscawen ST,Boscawen Street
8,Johnstone ST,Johnstone Street
9,Oliver ST,Oliver Street


In [47]:
clean_df = df[
    df["situation_number"].notna() & (df["situation_number"].str.strip() != "")
    & df["situation_name"].notna() & (df["situation_name"].str.strip() != "")
]
print(f"{len(clean_df)} of {len(df)} rows have both situation_number and situation_name")

# df_sample = clean_df.sample(n=10000, random_state=42)
df_sample = clean_df


90738 of 92010 rows have both situation_number and situation_name


In [ ]:
# Build a search address per row from what we have (no suburb available yet),
# using the abbreviation-expanded street name so it matches LINZ's full-form
# reference data more closely.
addresses = (
    df_sample["situation_number"] + " " + df_sample["situation_name_expanded"]
).str.strip()

results = gc.geocode_addresses(addresses.tolist())

df_sample["geocode_confidence"] = [r.confidence for r in results]
df_sample["longitude"] = [r.matched_address.shape_X if r.matched_address else None for r in results]
df_sample["latitude"] = [r.matched_address.shape_Y if r.matched_address else None for r in results]
df_sample["matched_suburb"] = [r.matched_address.suburb_locality if r.matched_address else None for r in results]
df_sample["matched_postcode"] = [r.matched_address.postcode if r.matched_address else None for r in results]

print(df_sample["geocode_confidence"].describe())


In [ ]:
# Sanity check: with the candidate pool restricted to Auckland, it should be
# structurally impossible for a match to land outside the region.
lon = df_sample["longitude"].astype(float)
lat = df_sample["latitude"].astype(float)
out_of_bounds = ~(lon.between(AKL_LON_MIN, AKL_LON_MAX) & lat.between(AKL_LAT_MIN, AKL_LAT_MAX))
print(f"matches outside Auckland bounding box: {out_of_bounds.sum()} of {len(df_sample)}")


In [70]:
df.info()

<class 'pandas.DataFrame'>
Index: 90738 entries, 0 to 92009
Data columns (total 52 columns):
 #   Column                            Non-Null Count  Dtype  
---  ------                            --------------  -----  
 0   valuation_roll_number             90738 non-null  str    
 1   valuation_number_assessment       90738 non-null  str    
 2   valuation_number_suffix           14016 non-null  str    
 3   sale_date                         90738 non-null  str    
 4   district_code                     90738 non-null  str    
 5   sale_type                         90737 non-null  str    
 6   sales_group                       90738 non-null  str    
 7   sale_tenure                       90738 non-null  str    
 8   price_value_relationship          90738 non-null  str    
 9   sale_price_gross                  90738 non-null  str    
 10  sale_price_net                    90738 non-null  str    
 11  sale_price_chattels               90738 non-null  str    
 12  sale_price_other    

In [51]:
df = df_sample

In [49]:
flood_plains = gpd.read_file("./hazard_data/flood_plains.geojson")
flood_sensitive = gpd.read_file("./hazard_data/flood_sensitive_area.geojson")
unitary_plan_zones = gpd.read_file("./hazard_data/Unitary_Plan_Base_Zone.geojson")


In [52]:
geocoded = df.dropna(subset=["longitude", "latitude"])
point_gdf = gpd.GeoDataFrame(
    geocoded[[]],
    geometry = gpd.points_from_xy(geocoded["longitude"], geocoded["latitude"]),
    crs="EPSG:4326",
)



In [53]:
flood_plains_matches = gpd.sjoin(point_gdf, flood_plains, how = "inner", predicate="within")
df["in_flood_plain"] = df.index.isin(flood_plains_matches.index)

In [54]:

flood_sensitive_matches = gpd.sjoin(point_gdf, flood_sensitive, how = "inner", predicate="within")
df["in_flood_sensitive_area"] = df.index.isin(flood_sensitive_matches.index)

In [55]:
unitary_zone = gpd.sjoin(point_gdf, unitary_plan_zones, how="left", predicate="within")
unitary_zone = unitary_zone[~unitary_zone.index.duplicated(keep="first")]  # in case of overlapping zone polygons
df["unitary_plan_zone"] = unitary_zone["ZONE"]

In [56]:
df["in_flood_plain"].describe()

count     90738
unique        2
top       False
freq      85816
Name: in_flood_plain, dtype: object

In [57]:
df["in_flood_sensitive_area"].describe()

count     90738
unique        2
top       False
freq      89056
Name: in_flood_sensitive_area, dtype: object

In [58]:
df["unitary_plan_zone"].describe()

count    60882.000000
mean        27.031389
std         18.800734
min          1.000000
25%         18.000000
50%         18.000000
75%         35.000000
max         69.000000
Name: unitary_plan_zone, dtype: float64

In [59]:
df["unitary_plan_zone"].unique()

array([60., nan, 18.,  8., 20., 19., 12., 16., 44., 33., 69., 17., 11.,
        4., 22., 32.,  7., 10., 51., 35., 46., 34.,  5., 30., 27., 40.,
       23., 31.,  3., 49.,  1., 63., 54., 68., 56., 43., 62., 26., 15.,
       55., 61., 52.])

In [60]:
points_nztm = point_gdf.to_crs("EPSG:2193")
flood_plains_nztm = flood_plains.to_crs("EPSG:2193")
flood_sensitive_nztm = flood_sensitive.to_crs("EPSG:2193")

In [61]:
flood_plains_nearest = gpd.sjoin_nearest(
    points_nztm, flood_plains_nztm, distance_col="dist_to_flood_plain_m"
)
flood_plains_nearest = flood_plains_nearest[~flood_plains_nearest.index.duplicated(keep="first")]
df["dist_to_flood_plain_m"] = flood_plains_nearest["dist_to_flood_plain_m"]

flood_sensitive_nearest = gpd.sjoin_nearest(
    points_nztm, flood_sensitive_nztm, distance_col="dist_to_flood_sensitive_area_m"
)
flood_sensitive_nearest = flood_sensitive_nearest[~flood_sensitive_nearest.index.duplicated(keep="first")]
df["dist_to_flood_sensitive_area_m"] = flood_sensitive_nearest["dist_to_flood_sensitive_area_m"]

print(df[["dist_to_flood_plain_m", "dist_to_flood_sensitive_area_m"]].describe())

       dist_to_flood_plain_m  dist_to_flood_sensitive_area_m
count           9.073800e+04                    9.073800e+04
mean            1.233983e+05                    1.301807e+05
std             2.589938e+05                    2.617708e+05
min             0.000000e+00                    0.000000e+00
25%             5.107920e+01                    6.103244e+02
50%             1.363123e+02                    2.782417e+03
75%             1.000056e+05                    1.172030e+05
max             1.158145e+06                    1.172334e+06


In [63]:
df[["dist_to_flood_plain_m", "dist_to_flood_sensitive_area_m"]].head(5)

,dist_to_flood_plain_m,dist_to_flood_sensitive_area_m
0,2.419102e+02,4.866407e+03
1,1.018463e+06,1.031195e+06
2,1.962920e+02,4.670897e+03
3,1.882070e+01,4.814777e+03
4,6.764028e+00,4.804595e+03


In [66]:
CACHE_PATH.parent.mkdir(parents=True, exist_ok=True)
df.to_parquet(CACHE_PATH)
print(f"Saved checkpoint to {CACHE_PATH}, shape={df.shape}")


Saved checkpoint to cache\dvrs_processed.parquet, shape=(90738, 52)


## Analysis

In [67]:
df_low = df_sample[df_sample["geocode_confidence"] < 0.5]
df_low[["situation_number", "situation_name", "geocode_confidence", "longitude", "latitude", "situation_name_expanded"]]

,situation_number,situation_name,geocode_confidence,longitude,latitude,situation_name_expanded
10283,15,Silica MW,0.487302,176.9226114,-39.5805906,Silica Mews
10403,34,Penehareti RI,0.490168,176.0341643833,-38.3945872167,Penehareti Rise
12541,1,Peart VW,0.414575,178.0322386,-38.6544899333,Peart VW
12542,7,Peart VW,0.415713,178.0330072,-38.6545728167,Peart VW
12669,24,Silica MW,0.484176,176.9227362,-39.5814314333,Silica Mews
...,...,...,...,...,...,...
91739,15,Yellow Pear LA,0.477658,174.8672128333,-37.02246435,Yellow Pear Lane
91740,14,Yellow Pear LA,0.488104,174.96999675,-37.05831145,Yellow Pear Lane
91742,16,Yellow Pear LA,0.474458,174.8666662667,-37.022057,Yellow Pear Lane
91856,6,Shortfin PL,0.476698,175.5217042833,-41.0265781167,Shortfin Place


In [68]:
sample = df.sample(n=1000, random_state=42).copy()

sample["address"] = (
    sample["situation_number"].fillna("") + " " + sample["situation_name_expanded"].fillna("")
).str.strip()

export_df = sample[["address", "geocode_confidence", "latitude", "longitude"]].reset_index(drop=True)

with pd.ExcelWriter("address_sample.xlsx", engine="openpyxl") as writer:
    for i in range(5):
        chunk = export_df.iloc[i * 200 : (i + 1) * 200]
        chunk.to_excel(writer, sheet_name=f"Sheet{i+1}", index=False)

print("Saved address_sample.xlsx")


Saved address_sample.xlsx


In [73]:
df[df["geocode_confidence"] > 0.6]

,valuation_roll_number,valuation_number_assessment,valuation_number_suffix,sale_date,district_code,sale_type,sales_group,sale_tenure,price_value_relationship,sale_price_gross,...,longitude,latitude,matched_suburb,matched_postcode,in_flood_plain,in_flood_sensitive_area,unitary_plan_zone,situation_name_expanded,dist_to_flood_plain_m,dist_to_flood_sensitive_area_m
0,1,2501710000,NaN,25022026,14,S,701,1,1,1514000,...,174.7063887333,-36.86568815,Point Chevalier,1022,False,False,60.0,Point Chevalier Road,2.419102e+02,4.866407e+03
1,1,2600010100,NaN,1042025,14,S,701,1,1,1515112,...,170.34350005,-45.8760796167,Mosgiel,9024,False,False,NaN,Montrose Street,1.018463e+06,1.031195e+06
2,1,4300000201,NaN,14022025,14,S,701,1,1,1622800,...,174.7028569,-36.8613287333,Point Chevalier,1022,False,False,60.0,St Michaels Avenue,1.962920e+02,4.670897e+03
3,1,4300110000,NaN,17012025,14,S,701,1,1,2105000,...,174.7008459833,-36.86233865,Point Chevalier,1022,False,False,18.0,St Michaels Avenue,1.882070e+01,4.814777e+03
4,1,4300270001,NaN,11112025,14,S,701,1,2,1875000,...,174.6988467667,-36.8636163333,Point Chevalier,1022,False,False,18.0,St Michaels Avenue,6.764028e+00,4.804595e+03
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
92005,38701,12900,NaN,24072025,14,S,1002,1,2,610000,...,173.290813185951,-41.2777725308956,Nelson,7010,False,False,NaN,Tasman Street,4.579296e+05,4.694704e+05
92006,38701,13100,NaN,8032025,14,S,1002,1,1,670000,...,173.8532665167,-39.4521156,Opunake,4616,False,False,NaN,Tasman Street,2.499408e+05,2.621737e+05
92007,38701,15200,NaN,23092025,14,S,1002,1,1,600000,...,174.8900347167,-37.19514775,Pukekohe,2120,False,False,18.0,Arnhem Place,1.135919e+02,7.992038e+01
92008,38701,20900,NaN,21042024,14,S,1002,1,1,693500,...,176.25125845,-38.1501478167,Glenholme,3010,False,False,NaN,Holland Street,1.456930e+05,1.566662e+05
